In [ ]:
# ── Setup: src/ on path, build df_acc through the merger, paths from config ──
import os, sys, re
from pathlib import Path
from collections import Counter
_p = os.getcwd()
while _p != os.path.dirname(_p):
    if os.path.isdir(os.path.join(_p, "src")):
        sys.path.insert(0, os.path.join(_p, "src")); break
    _p = os.path.dirname(_p)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from analysis_data import build_analysis_dataframe
from config import RESULTS_SEMI_DIR, CONFIG_DIRS

df_acc = build_analysis_dataframe(config="main_0.3")
topic_main = pd.read_csv(os.path.join(RESULTS_SEMI_DIR, CONFIG_DIRS["main_0.3"], "topic_info.csv"))

OUT_DIR = Path("results/cluster_appendix")
OUT_DIR.mkdir(parents=True, exist_ok=True)

In [8]:
# -------------------------
# 6. Load stopwords through stop_words.py
# -------------------------

from collections import Counter
import re

# Load stopwords
from stop_words import get_stop_words
stopwords = set(get_stop_words())

# Manual additions — artifacts and filler words only
# Manual additions — artifacts and filler words only
manual_exclude = {
    # Artifacts
    's', 'p', 'a', 'f', 'g', 'm',
    # Generic verb forms
    'skete', 'foretog', 'kører', 'køre', 'ramte', 'ramt',
    'nåede', 'standsede', 'standset', 'bemærkede', 'opdagede',
    'undgå', 'foretage',
    # Generic filler nouns
    'parterne', 'herunder', 'side', 'frem', 'hvilket',
    'uheldet', 'færdselsuheld', 'hinanden', 'ligeledes',
    'umiddelbart', 'hvorunder', 'involveret', 'opstod', 'impliceret',
    # Generic spatial/directional
    'øst', 'venstre', 'højre', 'nordgående', 'sydgående',
}


stopwords = stopwords | manual_exclude

In [9]:
# -------------------------
# 7. Quick sanity check
# -------------------------

TEXT_COL = "police_narrative"
TOPIC_COL = "assigned_topic"

print(f"Rows in df_acc: {len(df_acc):,}")
print(f"Unique accidents: {df_acc['accident_id'].nunique():,}")
print(f"Topics: {sorted(df_acc[TOPIC_COL].dropna().unique().astype(int))[:5]} ...")
print(f"Number of non-outlier topics: {(df_acc[TOPIC_COL] != -1).sum():,} assigned accidents")
print(f"Text column present: {TEXT_COL in df_acc.columns}")
print(f"Topic info rows: {len(topic_main):,}")

Rows in df_acc: 890,540
Unique accidents: 890,540
Topics: [-1, 0, 1, 2, 3] ...
Number of non-outlier topics: 600,004 assigned accidents
Text column present: True
Topic info rows: 42


In [10]:
PALETTE = {
    "blue_dark": "#1F4E79",
    "blue": "#2F75B5",
    "blue_light": "#9DC3E6",
    "gray_dark": "#595959",
    "gray": "#A6A6A6",
    "gray_light": "#D9E2F3",
    "grid": "#D9D9D9",
}

In [13]:
# Use whichever column name is correct in your df; you wrote both spellings
SUB = "encoded_accident_situation"   # the sub accident situation code
TOP = "assigned_topic"

# Optional: drop the BERTopic outlier bucket so it doesn't distort the range
d = df_acc[df_acc[TOP] != -1].copy()

# --- Per-topic dominant sub-situation and its concentration ------------------
def dominant(group):
    shares = group[SUB].value_counts(normalize=True)
    return pd.Series({
        "n_narratives": len(group),
        "dominant_sub": shares.index[0],
        "dominant_share": shares.iloc[0],        # purity of the topic
        "n_sub_situations": group[SUB].nunique(),
    })

per_topic = (
    d.groupby(TOP, group_keys=False)
     .apply(dominant)
     .sort_values("dominant_share", ascending=False)
)
per_topic["dominant_share"] = per_topic["dominant_share"].astype(float)

print(per_topic.to_string())

# --- The X–Y% range for the placeholder --------------------------------------
lo = per_topic["dominant_share"].min() * 100
hi = per_topic["dominant_share"].max() * 100
print(f"\nDominant sub-situation accounts for {lo:.0f}–{hi:.0f}% of each topic")

# --- 'most concentrate on a single sub-situation' ----------------------------
share_majority = (per_topic["dominant_share"] >= 0.5).mean()
print(f"{share_majority*100:.0f}% of topics have a sub-situation as an outright majority (>=50%)")

# --- 'a single sub-situation can be dominant in several topics' --------------
multi = (per_topic.groupby("dominant_sub")
                  .size()
                  .loc[lambda s: s > 1]
                  .sort_values(ascending=False))
print("\nSub-situations that are the dominant class in >1 topic:")
print(multi.to_string())


                n_narratives  dominant_sub  dominant_share  n_sub_situations
assigned_topic                                                              
40                     906.0         740.0        0.970199              15.0
18                    2542.0         140.0        0.966168              17.0
12                    5300.0         910.0        0.955283              37.0
23                    1545.0         140.0        0.953398              24.0
29                    1287.0         910.0        0.940948              23.0
24                    1526.0         170.0        0.903014              29.0
32                    1262.0         910.0        0.889065               8.0
27                    1361.0         140.0        0.853049              30.0
31                    1277.0         312.0        0.720439              11.0
9                     6850.0         140.0        0.719708              43.0
36                    1082.0         410.0        0.705176              14.0

/tmp/ipykernel_3754260/3661469025.py:20: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(dominant)


In [21]:
keep = per_topic[per_topic["dominant_share"] >= 0.7].copy()

lo = keep["dominant_share"].min() * 100
hi = keep["dominant_share"].max() * 100
print(f"{len(keep)} mechanism topics; dominant sub-situation = {lo:.0f}–{hi:.0f}% of topic")

# sub-situations dominant in >1 of the retained topics
multi = (keep.groupby("dominant_sub")
             .apply(lambda g: list(g.index))
             .loc[lambda s: s.str.len() > 1])
print(multi)

11 mechanism topics; dominant sub-situation = 71–97% of topic
dominant_sub
140.0    [18, 23, 27, 9]
910.0       [12, 29, 32]
dtype: object


/tmp/ipykernel_3754260/912111126.py:9: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda g: list(g.index))


In [13]:
# -------------------------
# Helper functions
# -------------------------

def parse_representation(value):
    """Parse BERTopic representation stored as list-like string or plain text."""
    if pd.isna(value):
        return []
    if isinstance(value, list):
        return value
    try:
        parsed = ast.literal_eval(value)
        if isinstance(parsed, list):
            return [str(x) for x in parsed]
    except Exception:
        pass
    return [x.strip() for x in str(value).replace(";", ",").split(",") if x.strip()]


def tokenize(text):
    text = str(text).lower()
    words = re.findall(r"\b[a-zæøå]+\b", text)
    return [w for w in words if w not in stopwords and len(w) > 2]


def top_words_for_cluster(sub, n=10):
    tokens = []
    for txt in sub[TEXT_COL].dropna():
        tokens.extend(tokenize(txt))
    return Counter(tokens).most_common(n)


def top_bigrams_for_cluster(sub, n=10):
    bigrams = []
    for txt in sub[TEXT_COL].dropna():
        toks = tokenize(txt)
        bigrams.extend(zip(toks, toks[1:]))
    counts = Counter([" ".join(bg) for bg in bigrams])
    return counts.most_common(n)


def profile_lift(df_acc, sub, candidate_cols=None, min_topic_count=20, top_n=10):
    """
    Finds categories overrepresented in the topic compared with corpus baseline.
    Returns rows like: variable, category, topic_share, corpus_share, lift, topic_count.
    """
    if candidate_cols is None:
        candidate_cols = [
            "report_category",
            "VEJKATEGORI",
            "LYS",
            "VEJRFORHOLD",
            "kommune_group",
            "sprit_flag",
            "has_cyclist",
            "element_1_age_group",
            "element_2_age_group",
            "element_1_type",
            "element_2_type",
        ]

    rows = []

    for col in candidate_cols:
        if col not in df_acc.columns:
            continue

        topic_counts = sub[col].value_counts(dropna=False)
        corpus_counts = df_acc[col].value_counts(dropna=False)

        topic_shares = topic_counts / len(sub)
        corpus_shares = corpus_counts / len(df_acc)

        for cat, topic_share in topic_shares.items():
            topic_count = int(topic_counts.loc[cat])
            if topic_count < min_topic_count:
                continue

            corpus_share = float(corpus_shares.get(cat, 0))
            if corpus_share <= 0:
                continue

            rows.append({
                "variable": col,
                "category": str(cat),
                "topic_share": float(topic_share),
                "corpus_share": corpus_share,
                "lift": float(topic_share / corpus_share),
                "topic_count": topic_count,
            })

    out = pd.DataFrame(rows)
    if out.empty:
        return out

    return (
        out.sort_values(["lift", "topic_share"], ascending=False)
           .head(top_n)
           .reset_index(drop=True)
    )


def get_topic_row(topic_id):
    if "Topic" not in topic_main.columns:
        return None
    rows = topic_main[topic_main["Topic"] == topic_id]
    if rows.empty:
        return None
    return rows.iloc[0]

In [17]:
PALETTE = {
    "blue_dark": "#1F4E79",
    "blue": "#2F75B5",
    "blue_light": "#9DC3E6",
    "gray_dark": "#595959",
    "gray": "#A6A6A6",
    "gray_light": "#D9E2F3",
    "grid": "#D9D9D9",
}


def style_axis(ax):
    ax.grid(axis="x", alpha=0.35, color=PALETTE["grid"])
    ax.tick_params(axis="both", labelsize=8)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)


def save_barh_plot(labels, values, path, color, xlabel):
    fig, ax = plt.subplots(figsize=(4.2, 3.0))

    if labels:
        ax.barh(labels[::-1], values[::-1], color=color)

    ax.set_xlabel(xlabel, fontsize=9)
    style_axis(ax)

    plt.tight_layout()
    plt.savefig(path, dpi=250, bbox_inches="tight")
    plt.close(fig)


def plot_cluster_individual(topic_id, df_acc, topic_main):
    sub = df_acc[df_acc[TOPIC_COL] == topic_id].copy()
    if sub.empty:
        return None

    topic_row = get_topic_row(topic_id)

    if topic_row is not None and "Name" in topic_row:
        name = str(topic_row["Name"])
    else:
        name = f"Topic {topic_id}"

    words = top_words_for_cluster(sub, n=10)
    bigrams = top_bigrams_for_cluster(sub, n=10)
    lifts = profile_lift(
        df_acc,
        sub,
        min_topic_count=max(10, int(len(sub) * 0.01)),
        top_n=10,
    )

    topic_dir = OUT_DIR / f"cluster_{topic_id:02d}"
    topic_dir.mkdir(parents=True, exist_ok=True)

    # 1. Top words
    word_labels = [w for w, _ in words]
    word_values = [v for _, v in words]
    words_path = topic_dir / "top_words.png"

    save_barh_plot(
        word_labels,
        word_values,
        words_path,
        color=PALETTE["blue_dark"],
        xlabel="Count",
    )

    # 2. Top bigrams
    bigram_labels = [b for b, _ in bigrams]
    bigram_values = [v for _, v in bigrams]
    bigrams_path = topic_dir / "top_bigrams.png"

    save_barh_plot(
        bigram_labels,
        bigram_values,
        bigrams_path,
        color=PALETTE["blue"],
        xlabel="Count",
    )

    # 3. Profile lift
    lift_path = topic_dir / "profile_lift.png"

    fig, ax = plt.subplots(figsize=(4.2, 3.0))

    if not lifts.empty:
        lift_labels = [
            f"{r.variable}: {r.category}"
            for r in lifts.itertuples(index=False)
        ]
        lift_values = lifts["lift"].values

        ax.barh(lift_labels[::-1], lift_values[::-1], color=PALETTE["gray_dark"])
        ax.axvline(1, color=PALETTE["blue_dark"], linewidth=0.9)

    ax.set_xlabel("Lift relative to corpus", fontsize=9)
    style_axis(ax)

    plt.tight_layout()
    plt.savefig(lift_path, dpi=250, bbox_inches="tight")
    plt.close(fig)

    return {
        "topic": topic_id,
        "name": name,
        "count": len(sub),
        "top_words": words,
        "top_bigrams": bigrams,
        "profile_lift": lifts,
        "top_words_path": str(words_path),
        "top_bigrams_path": str(bigrams_path),
        "profile_lift_path": str(lift_path),
    }

In [15]:
# -------------------------
# Short draft summary
# -------------------------

def make_summary(result):
    topic = result["topic"]
    name = result["name"]
    count = result["count"]

    word_terms = [w for w, _ in result["top_words"][:5]]
    bigram_terms = [b for b, _ in result["top_bigrams"][:4]]

    lifts = result["profile_lift"]
    if not lifts.empty:
        top_profile = lifts.iloc[0]
        profile_sentence = (
            f"The strongest structured-data signal is overrepresentation of "
            f"{top_profile['category']} in {top_profile['variable']} "
            f"(lift {top_profile['lift']:.2f})."
        )
    else:
        profile_sentence = (
            "No single structured-data category is strongly overrepresented under the selected thresholds."
        )

    return (
        f"Cluster {topic} contains {count:,} accidents and is provisionally labelled "
        f"'{name}'. The filtered word profile is led by {', '.join(word_terms)}, "
        f"while the main bigrams include {', '.join(bigram_terms)}. Together, these "
        f"terms suggest that the cluster captures a recurring narrative pattern rather "
        f"than only the BERTopic representative keywords. {profile_sentence}"
    )

In [18]:
cluster_results = []
summary_rows = []

topic_ids = sorted(
    t for t in df_acc[TOPIC_COL].dropna().unique().astype(int)
    if t != -1
)

for topic_id in topic_ids:
    result = plot_cluster_individual(topic_id, df_acc, topic_main)
    if result is None:
        continue

    summary = make_summary(result)

    topic_dir = OUT_DIR / f"cluster_{topic_id:02d}"

    with open(topic_dir / "summary.txt", "w", encoding="utf-8") as f:
        f.write(summary)

    cluster_results.append(result)

    summary_rows.append({
        "topic": topic_id,
        "name": result["name"],
        "count": result["count"],
        "top_words": "; ".join([w for w, _ in result["top_words"]]),
        "top_bigrams": "; ".join([b for b, _ in result["top_bigrams"]]),
        "summary": summary,
        "top_words_path": result["top_words_path"],
        "top_bigrams_path": result["top_bigrams_path"],
        "profile_lift_path": result["profile_lift_path"],
    })

cluster_summaries = pd.DataFrame(summary_rows)
cluster_summaries.to_csv(OUT_DIR / "cluster_summaries.csv", index=False)

cluster_summaries.head()

,topic,name,count,top_words,top_bigrams,summary,top_words_path,top_bigrams_path,profile_lift_path
0,0,0_vigepligt_ubetinget_ubetinget vigepligt_vens...,246857,krydset; vigepligt; påkørt; holdt; venstresvin...,ubetinget vigepligt; ubetingede vigepligt; hol...,"Cluster 0 contains 246,857 accidents and is pr...",results/cluster_appendix/cluster_00/top_words.png,results/cluster_appendix/cluster_00/top_bigram...,results/cluster_appendix/cluster_00/profile_li...
1,1,1_retn_ramte_foretog_frem,142352,krydset; holdt; påkørt; vognbane; mistede; vig...,mistede herredømmet; holdt stille; ubetinget v...,"Cluster 1 contains 142,352 accidents and is pr...",results/cluster_appendix/cluster_01/top_words.png,results/cluster_appendix/cluster_01/top_bigram...,results/cluster_appendix/cluster_01/profile_li...
2,2,2_rødt_lys_rødt lys_grønt,35419,krydset; lys; rødt; holdt; grønt; påkørt; bagf...,rødt lys; grønt lys; holdt rødt; lys krydset; ...,"Cluster 2 contains 35,419 accidents and is pro...",results/cluster_appendix/cluster_02/top_words.png,results/cluster_appendix/cluster_02/top_bigram...,results/cluster_appendix/cluster_02/profile_li...
3,3,3_støder_foretager_påkører_kører,29676,foretager; påkører; støder; venstresving; park...,foretager venstresving; påkører parkerede; rød...,"Cluster 3 contains 29,676 accidents and is pro...",results/cluster_appendix/cluster_03/top_words.png,results/cluster_appendix/cluster_03/top_bigram...,results/cluster_appendix/cluster_03/profile_li...
4,4,4_kort_ringe kort_kort holdt_kort lastbil,17455,kort; holdt; vognbane; bagfra; køretøj; misted...,mistede herredømmet; holdt stille; holdt parke...,"Cluster 4 contains 17,455 accidents and is pro...",results/cluster_appendix/cluster_04/top_words.png,results/cluster_appendix/cluster_04/top_bigram...,results/cluster_appendix/cluster_04/profile_li...
